# Branch Smoke Checks: APH/PPH Split

This notebook does quick, rough validation checks for branch changes around the APH/PPH split and hemorrhage logic.

Checks covered:
1. APH and PPH incidence are in the ballpark of artifact expectations
2. Only severe hemorrhage cases die from hemorrhage
3. Misoprostol affects postpartum hemorrhage incidence
4. HemoglobinRiskEffect: low hemoglobin simulants have higher APH/PPH incidence
5. PPH and APH shift data is negative (hemoglobin decreases after hemorrhage)
6. 0-6 week postpartum hemoglobin reflects hemorrhage shifts
7. Postpartum hemoglobin: 6w-9m values are drawn from non-pregnant distribution with hemorrhage shifts

In [ ]:
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd

import vivarium_gates_mncnh
from vivarium import Artifact, InteractiveContext
from vivarium.framework.configuration import build_model_specification

from vivarium_gates_mncnh.constants.data_keys import MATERNAL_HEMORRHAGE
from vivarium_gates_mncnh.constants.data_values import (
    ANEMIA_THRESHOLDS,
    COLUMNS,
    HEMORRHAGE_SEVERITY,
    NON_PREGNANT_ANEMIA_THRESHOLDS,
    SIMULATION_EVENT_NAMES,
)

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

In [ ]:
# Build a small interactive simulation for quick checks
model_spec_path = Path(vivarium_gates_mncnh.__file__).parent / 'model_specifications/model_spec.yaml'
base_spec = build_model_specification(model_spec_path)

DRAW = int(base_spec.configuration.input_data.input_draw_number)
POP_SIZE = 60_000

SIMULATION_STEPS = [
    SIMULATION_EVENT_NAMES.FIRST_TRIMESTER_ANC,
    SIMULATION_EVENT_NAMES.LATER_PREGNANCY_SCREENING,
    SIMULATION_EVENT_NAMES.LATER_PREGNANCY_INTERVENTION,
    SIMULATION_EVENT_NAMES.LATER_PREGNANCY_VISIT_TIMING,
    SIMULATION_EVENT_NAMES.ULTRASOUND,
    SIMULATION_EVENT_NAMES.ANTEPARTUM_HEMORRHAGE,
    SIMULATION_EVENT_NAMES.DELIVERY_FACILITY,
    SIMULATION_EVENT_NAMES.AZITHROMYCIN_ACCESS,
    SIMULATION_EVENT_NAMES.MISOPROSTOL_ACCESS,
    SIMULATION_EVENT_NAMES.CPAP_ACCESS,
    SIMULATION_EVENT_NAMES.ACS_ACCESS,
    SIMULATION_EVENT_NAMES.ANTIBIOTICS_ACCESS,
    SIMULATION_EVENT_NAMES.PROBIOTICS_ACCESS,
    SIMULATION_EVENT_NAMES.OBSTRUCTED_LABOR,
    SIMULATION_EVENT_NAMES.POSTPARTUM_HEMORRHAGE,
    SIMULATION_EVENT_NAMES.MATERNAL_SEPSIS,
    SIMULATION_EVENT_NAMES.RESIDUAL_MATERNAL_DISORDERS,
    SIMULATION_EVENT_NAMES.ABORTION_MISCARRIAGE_ECTOPIC_PREGNANCY,
    SIMULATION_EVENT_NAMES.MORTALITY,
    SIMULATION_EVENT_NAMES.EARLY_NEONATAL_MORTALITY,
    SIMULATION_EVENT_NAMES.LATE_NEONATAL_MORTALITY,
    SIMULATION_EVENT_NAMES.POSTPARTUM_HEMOGLOBIN_NINE_MONTH,
    SIMULATION_EVENT_NAMES.POSTPARTUM_DEPRESSION,
]

STEP_MAPPER = {name: i + 1 for i, name in enumerate(SIMULATION_STEPS)}

def make_spec(scenario: str = 'baseline', pop_size: int = POP_SIZE):
    spec = deepcopy(base_spec)
    spec.configuration.population.population_size = pop_size
    spec.configuration.intervention.scenario = scenario
    return spec

def make_sim(scenario: str = 'baseline', pop_size: int = POP_SIZE):
    return InteractiveContext(make_spec(scenario=scenario, pop_size=pop_size))

def run_to_step(sim: InteractiveContext, step_name: str):
    steps_needed = STEP_MAPPER[step_name] - sim._clock.step_index
    assert steps_needed > 0, f"Sim is already at or past step '{step_name}' (current index: {sim._clock.step_index})"
    sim.take_steps(steps_needed)
    return sim

artifact = Artifact(base_spec.configuration.input_data.artifact_path)
draw_col = f'draw_{DRAW}'
print('Using artifact:', base_spec.configuration.input_data.artifact_path)
print('Using draw column:', draw_col)
print('Population size:', POP_SIZE)

## 1) APH/PPH incidence and severity vs artifact (rough)

In [ ]:
def get_art_by_age(art_key: str, sim_year: int, sim_location: str, sim_sex: str = 'Female') -> pd.DataFrame:
    art = artifact.load(art_key).reset_index()

    # Filter progressively; if a filter wipes everything out, keep previous data
    cur = art.copy()
    if 'location' in cur.columns:
        next_df = cur[cur['location'] == sim_location]
        if not next_df.empty:
            cur = next_df
    if 'sex' in cur.columns:
        next_df = cur[cur['sex'] == sim_sex]
        if not next_df.empty:
            cur = next_df
    if {'year_start', 'year_end'}.issubset(cur.columns):
        next_df = cur[(cur['year_start'] <= sim_year) & (sim_year < cur['year_end'])]
        if not next_df.empty:
            cur = next_df

    grouped = cur.groupby(['age_start', 'age_end'], as_index=False)[draw_col].mean()
    return grouped.sort_values(['age_start', 'age_end']).reset_index(drop=True)

def map_ages_to_art_values(ages: pd.Series, art_by_age: pd.DataFrame) -> pd.Series:
    intervals = pd.IntervalIndex.from_arrays(
        art_by_age['age_start'].astype(float),
        art_by_age['age_end'].astype(float),
        closed='left',
    )
    idx = intervals.get_indexer(ages.astype(float))
    out = pd.Series(np.nan, index=ages.index, dtype=float)
    matched = idx >= 0
    if matched.any():
        out.loc[matched] = art_by_age.iloc[idx[matched]][draw_col].to_numpy()
    return out

def weighted_incidence_from_artifact(
    pop: pd.DataFrame,
    art_key: str,
    eligible_mask: pd.Series,
    sim_year: int,
    sim_location: str,
    sim_sex: str = 'Female',
) -> float:
    art_by_age = get_art_by_age(art_key, sim_year=sim_year, sim_location=sim_location, sim_sex=sim_sex)
    eligible_ages = pop.loc[eligible_mask, COLUMNS.MOTHER_AGE]
    values = map_ages_to_art_values(eligible_ages, art_by_age)
    matched = values.notna()
    return float(values[matched].mean()) if matched.any() else np.nan

sim_year = int(base_spec.configuration.time.start.year)

# Single run for both APH and PPH checks (up to PPH step)
sim = make_sim('baseline')
run_to_step(sim, SIMULATION_EVENT_NAMES.POSTPARTUM_HEMORRHAGE)
pop = sim.get_population([
    COLUMNS.MOTHER_AGE,
    COLUMNS.LOCATION,
    COLUMNS.PREGNANCY_OUTCOME,
    COLUMNS.ANTEPARTUM_HEMORRHAGE,
    COLUMNS.POSTPARTUM_HEMORRHAGE,
])

sim_location = pop[COLUMNS.LOCATION].mode().iloc[0]

# Denominators must match component eligibility rules
aph_eligible = pop[COLUMNS.PREGNANCY_OUTCOME] != 'invalid'
pph_eligible = pop[COLUMNS.PREGNANCY_OUTCOME].isin(['live_birth', 'stillbirth'])

# Sim incidence
aph_cases = pop[COLUMNS.ANTEPARTUM_HEMORRHAGE].astype(bool)
pph_cases = pop[COLUMNS.POSTPARTUM_HEMORRHAGE].astype(bool)

aph_incidence_sim = float(aph_cases[aph_eligible].mean()) if int(aph_eligible.sum()) else np.nan
pph_incidence_sim = float(pph_cases[pph_eligible].mean()) if int(pph_eligible.sum()) else np.nan

# Artifact incidence weighted to simulated eligible age composition
aph_incidence_art = weighted_incidence_from_artifact(
    pop,
    MATERNAL_HEMORRHAGE.APH_INCIDENCE_RISK,
    aph_eligible,
    sim_year=sim_year,
    sim_location=sim_location,
)
pph_incidence_art = weighted_incidence_from_artifact(
    pop,
    MATERNAL_HEMORRHAGE.PPH_INCIDENCE_RISK,
    pph_eligible,
    sim_year=sim_year,
    sim_location=sim_location,
)

check1 = pd.DataFrame([
    {
        'cause': 'antepartum_hemorrhage',
        'sim_incidence': aph_incidence_sim,
        'artifact_incidence': aph_incidence_art,
        'incidence_ratio_sim_over_artifact': aph_incidence_sim / aph_incidence_art if aph_incidence_art else np.nan,
    },
    {
        'cause': 'postpartum_hemorrhage',
        'sim_incidence': pph_incidence_sim,
        'artifact_incidence': pph_incidence_art,
        'incidence_ratio_sim_over_artifact': pph_incidence_sim / pph_incidence_art if pph_incidence_art else np.nan,
    },
])

check1

## 2) Only severe hemorrhage cases die from hemorrhage

In [ ]:
sim_mort = make_sim('baseline')
run_to_step(sim_mort, SIMULATION_EVENT_NAMES.MORTALITY)

APH_SEVERITY_COL = f"{COLUMNS.ANTEPARTUM_HEMORRHAGE}_severity"
PPH_SEVERITY_COL = f"{COLUMNS.POSTPARTUM_HEMORRHAGE}_severity"

mort_pop = sim_mort.get_population([
    COLUMNS.MOTHER_CAUSE_OF_DEATH,
    APH_SEVERITY_COL,
    PPH_SEVERITY_COL,
])

hemo_cod = [COLUMNS.ANTEPARTUM_HEMORRHAGE, COLUMNS.POSTPARTUM_HEMORRHAGE]
dead_hemo = mort_pop[mort_pop[COLUMNS.MOTHER_CAUSE_OF_DEATH].isin(hemo_cod)]

bad_aph = dead_hemo[
    (dead_hemo[COLUMNS.MOTHER_CAUSE_OF_DEATH] == COLUMNS.ANTEPARTUM_HEMORRHAGE)
    & (dead_hemo[APH_SEVERITY_COL] != HEMORRHAGE_SEVERITY.SEVERE)
]
bad_pph = dead_hemo[
    (dead_hemo[COLUMNS.MOTHER_CAUSE_OF_DEATH] == COLUMNS.POSTPARTUM_HEMORRHAGE)
    & (dead_hemo[PPH_SEVERITY_COL] != HEMORRHAGE_SEVERITY.SEVERE)
]

assert len(bad_aph) == 0, f'{len(bad_aph)} non-severe APH deaths found'
assert len(bad_pph) == 0, f'{len(bad_pph)} non-severe PPH deaths found'
print(f'PASSED: all {len(dead_hemo)} hemorrhage deaths were severe')

## 3) Misoprostol affects postpartum hemorrhage incidence

In [ ]:
def pph_summary_for_scenario(scenario: str) -> pd.DataFrame:
    sim = make_sim(scenario)
    run_to_step(sim, SIMULATION_EVENT_NAMES.POSTPARTUM_HEMORRHAGE)
    pop = sim.get_population([
        COLUMNS.POSTPARTUM_HEMORRHAGE,
        COLUMNS.MISOPROSTOL_AVAILABLE,
        COLUMNS.DELIVERY_FACILITY_TYPE,
    ])

    out = []
    overall = float(pop[COLUMNS.POSTPARTUM_HEMORRHAGE].mean())
    out.append({'scenario': scenario, 'group': 'overall', 'pph_incidence': overall, 'n': len(pop)})

    for value in [True, False]:
        sub = pop[pop[COLUMNS.MISOPROSTOL_AVAILABLE] == value]
        if len(sub) == 0:
            continue
        out.append({
            'scenario': scenario,
            'group': f'misoprostol_available={value}',
            'pph_incidence': float(sub[COLUMNS.POSTPARTUM_HEMORRHAGE].mean()),
            'n': len(sub),
        })

    home = pop[pop[COLUMNS.DELIVERY_FACILITY_TYPE] == 'home']
    if len(home):
        out.append({
            'scenario': scenario,
            'group': 'home_only',
            'pph_incidence': float(home[COLUMNS.POSTPARTUM_HEMORRHAGE].mean()),
            'n': len(home),
        })

    return pd.DataFrame(out)

baseline_pph = pph_summary_for_scenario('baseline')
miso_vv_pph = pph_summary_for_scenario('misoprostol_vv')

check4 = pd.concat([baseline_pph, miso_vv_pph], ignore_index=True)
check4

In [ ]:
# Directional check: compare home-birth PPH incidence across scenarios.
# Misoprostol is only available to home births so that is the right comparison group.
baseline_home = float(check4.query("scenario == 'baseline' and group == 'home_only'")['pph_incidence'].iloc[0])
miso_home = float(check4.query("scenario == 'misoprostol_vv' and group == 'home_only'")['pph_incidence'].iloc[0])

pd.DataFrame([
    {'metric': 'baseline_home_birth_pph_incidence', 'value': baseline_home},
    {'metric': 'misoprostol_vv_home_birth_pph_incidence', 'value': miso_home},
    {'metric': 'directional_pass', 'value': miso_home < baseline_home},
])

## 4) HemoglobinRiskEffect modifies APH/PPH incidence

Higher hemoglobin is protective (RR < 1 at high exposure). Expected direction: low-hemoglobin tertile has higher APH and PPH incidence than the high-hemoglobin tertile.

In [ ]:
# Reuse sim from check 1 (already run to PPH step)
hgb_pop = sim.get_population([
    COLUMNS.HEMOGLOBIN_EXPOSURE,
    COLUMNS.PREGNANCY_OUTCOME,
    COLUMNS.ANTEPARTUM_HEMORRHAGE,
    COLUMNS.POSTPARTUM_HEMORRHAGE,
])

aph_elig = hgb_pop[COLUMNS.PREGNANCY_OUTCOME] != 'invalid'
pph_elig = hgb_pop[COLUMNS.PREGNANCY_OUTCOME].isin(['live_birth', 'stillbirth'])

hgb_pop['hgb_tertile'] = pd.qcut(hgb_pop[COLUMNS.HEMOGLOBIN_EXPOSURE], q=3, labels=['low', 'mid', 'high'])

rows = []
for tertile in ['low', 'mid', 'high']:
    t_mask = hgb_pop['hgb_tertile'] == tertile
    rows.append({
        'hgb_tertile': tertile,
        'aph_incidence': float(hgb_pop.loc[t_mask & aph_elig, COLUMNS.ANTEPARTUM_HEMORRHAGE].astype(bool).mean()),
        'pph_incidence': float(hgb_pop.loc[t_mask & pph_elig, COLUMNS.POSTPARTUM_HEMORRHAGE].astype(bool).mean()),
    })

check4_hgb = pd.DataFrame(rows)

low_aph = check4_hgb.loc[check4_hgb['hgb_tertile'] == 'low', 'aph_incidence'].iloc[0]
high_aph = check4_hgb.loc[check4_hgb['hgb_tertile'] == 'high', 'aph_incidence'].iloc[0]
low_pph = check4_hgb.loc[check4_hgb['hgb_tertile'] == 'low', 'pph_incidence'].iloc[0]
high_pph = check4_hgb.loc[check4_hgb['hgb_tertile'] == 'high', 'pph_incidence'].iloc[0]

check4_hgb['directional_pass'] = None
check4_hgb.loc[check4_hgb['hgb_tertile'] == 'high', 'directional_pass'] = (low_aph > high_aph) and (low_pph > high_pph)

check4_hgb

## 5) PPH and APH shift data is negative (hemoglobin decreases after hemorrhage)

Verify that the hemorrhage hemoglobin shift values loaded from the artifact are negative, confirming that hemorrhage reduces hemoglobin.

In [ ]:
from vivarium_gates_mncnh.constants.data_keys import HEMORRHAGE_HEMOGLOBIN_SHIFT

# Load shift values from the artifact
pph_shift_0_6w = float(artifact.load(HEMORRHAGE_HEMOGLOBIN_SHIFT.PPH_SHIFT_0_6W)[draw_col].iloc[0])
aph_shift_0_6w = float(artifact.load(HEMORRHAGE_HEMOGLOBIN_SHIFT.APH_SHIFT_0_6W)[draw_col].iloc[0])
pph_shift_6w_9m = float(artifact.load(HEMORRHAGE_HEMOGLOBIN_SHIFT.PPH_SHIFT_6W_9M)[draw_col].iloc[0])
aph_shift_6w_9m = float(artifact.load(HEMORRHAGE_HEMOGLOBIN_SHIFT.APH_SHIFT_6W_9M)[draw_col].iloc[0])

# CHECK 5a: PPH 0-6w shift is negative
assert pph_shift_0_6w < 0, f"Expected PPH 0-6w shift to be negative, got {pph_shift_0_6w}"
print(f"PASSED 5a: PPH 0-6w shift is negative ({pph_shift_0_6w:.2f} g/L)")

# CHECK 5b: APH 0-6w shift is negative
assert aph_shift_0_6w < 0, f"Expected APH 0-6w shift to be negative, got {aph_shift_0_6w}"
print(f"PASSED 5b: APH 0-6w shift is negative ({aph_shift_0_6w:.2f} g/L)")

# CHECK 5c: PPH 6w-9m shift is negative
assert pph_shift_6w_9m < 0, f"Expected PPH 6w-9m shift to be negative, got {pph_shift_6w_9m}"
print(f"PASSED 5c: PPH 6w-9m shift is negative ({pph_shift_6w_9m:.2f} g/L)")

# CHECK 5d: APH 6w-9m shift is negative
assert aph_shift_6w_9m < 0, f"Expected APH 6w-9m shift to be negative, got {aph_shift_6w_9m}"
print(f"PASSED 5d: APH 6w-9m shift is negative ({aph_shift_6w_9m:.2f} g/L)")

## 6) 0-6 week postpartum hemoglobin reflects hemorrhage shifts

At the `early_neonatal_mortality` event the component applies PPH and APH shifts to the pregnancy hemoglobin for hemorrhage cases. Expected:
- Simulants with PPH or APH have lower hemoglobin than those without hemorrhage.
- All hemoglobin values are non-negative.

In [ ]:
APH_SEVERITY_COL = f"{COLUMNS.ANTEPARTUM_HEMORRHAGE}_severity"
PPH_SEVERITY_COL = f"{COLUMNS.POSTPARTUM_HEMORRHAGE}_severity"

# Run sim to early neonatal mortality (0-6 week postpartum)
sim_6w = make_sim('baseline')
run_to_step(sim_6w, SIMULATION_EVENT_NAMES.EARLY_NEONATAL_MORTALITY)

pop_6w = sim_6w.get_population([
    COLUMNS.HEMOGLOBIN_EXPOSURE,
    COLUMNS.PREGNANCY_OUTCOME,
    COLUMNS.POSTPARTUM_HEMORRHAGE,
    COLUMNS.ANTEPARTUM_HEMORRHAGE,
])

full_term_6w = pop_6w[pop_6w[COLUMNS.PREGNANCY_OUTCOME].isin(['live_birth', 'stillbirth'])]

# CHECK 6a: all hemoglobin values are non-negative
assert (full_term_6w[COLUMNS.HEMOGLOBIN_EXPOSURE] >= 0).all(), "Found negative hemoglobin values at 6w"
print("PASSED 6a: all 0-6w hemoglobin values are >= 0")

# CHECK 6b: directional - hemorrhage cases have lower mean 6w hemoglobin
no_hemorrhage_6w = full_term_6w[
    (full_term_6w[COLUMNS.POSTPARTUM_HEMORRHAGE] == False)
    & (full_term_6w[COLUMNS.ANTEPARTUM_HEMORRHAGE] == False)
]
any_hemorrhage_6w = full_term_6w[
    (full_term_6w[COLUMNS.POSTPARTUM_HEMORRHAGE] == True)
    | (full_term_6w[COLUMNS.ANTEPARTUM_HEMORRHAGE] == True)
]

mean_no_hem_6w = no_hemorrhage_6w[COLUMNS.HEMOGLOBIN_EXPOSURE].mean()
mean_any_hem_6w = any_hemorrhage_6w[COLUMNS.HEMOGLOBIN_EXPOSURE].mean() if len(any_hemorrhage_6w) else np.nan

if len(any_hemorrhage_6w) > 0:
    assert mean_any_hem_6w < mean_no_hem_6w, (
        f"Expected hemorrhage cases to have lower 6w hemoglobin: "
        f"hemorrhage mean={mean_any_hem_6w:.1f}, no hemorrhage mean={mean_no_hem_6w:.1f}"
    )
    print(f"PASSED 6b: hemorrhage cases have lower 6w hemoglobin ({mean_any_hem_6w:.1f}) vs no hemorrhage ({mean_no_hem_6w:.1f})")
else:
    print("SKIPPED 6b: no hemorrhage cases in this run")

# Summary table
rows_6w = []
for label, subset in [('no_hemorrhage', no_hemorrhage_6w), ('pph_only', full_term_6w[
    (full_term_6w[COLUMNS.POSTPARTUM_HEMORRHAGE] == True) & (full_term_6w[COLUMNS.ANTEPARTUM_HEMORRHAGE] == False)
]), ('aph_only', full_term_6w[
    (full_term_6w[COLUMNS.POSTPARTUM_HEMORRHAGE] == False) & (full_term_6w[COLUMNS.ANTEPARTUM_HEMORRHAGE] == True)
]), ('both', full_term_6w[
    (full_term_6w[COLUMNS.POSTPARTUM_HEMORRHAGE] == True) & (full_term_6w[COLUMNS.ANTEPARTUM_HEMORRHAGE] == True)
])]:
    if len(subset) == 0:
        continue
    rows_6w.append({
        'group': label,
        'n': len(subset),
        'mean_6w_hgb': subset[COLUMNS.HEMOGLOBIN_EXPOSURE].mean(),
    })

summary_6w = pd.DataFrame(rows_6w)
summary_6w

## 7) 6w-9m postpartum hemoglobin from non-pregnant distribution with hemorrhage shifts

At the `postpartum_hemoglobin_nine_month` event the component draws a new hemoglobin from the non-pregnant distribution and applies PPH and APH shifts. Expected:
- All full-term mothers have a non-NaN 9-month hemoglobin value.
- Simulants with PPH or APH have lower 9-month hemoglobin than those without hemorrhage.
- 9-month hemoglobin values are non-negative and distinct from pregnancy hemoglobin.

In [ ]:
# Reuse sim_6w from check 6 — capture pre-9m hemoglobin (already has 0-6w shifts)
pre_9m_hgb = sim_6w.get_population([COLUMNS.HEMOGLOBIN_EXPOSURE])[COLUMNS.HEMOGLOBIN_EXPOSURE].copy()

# Continue to the 9-month event
run_to_step(sim_6w, SIMULATION_EVENT_NAMES.POSTPARTUM_HEMOGLOBIN_NINE_MONTH)

pop_9m = sim_6w.get_population([
    COLUMNS.HEMOGLOBIN_EXPOSURE,
    COLUMNS.PREGNANCY_OUTCOME,
    COLUMNS.POSTPARTUM_HEMORRHAGE,
    COLUMNS.ANTEPARTUM_HEMORRHAGE,
])

full_term_9m = pop_9m[pop_9m[COLUMNS.PREGNANCY_OUTCOME].isin(['live_birth', 'stillbirth'])]

# CHECK 7a: all full-term mothers have non-NaN hemoglobin at 9m
n_9m_notna = full_term_9m[COLUMNS.HEMOGLOBIN_EXPOSURE].notna().sum()
assert n_9m_notna == len(full_term_9m), (
    f"Expected all {len(full_term_9m)} full-term mothers to have hemoglobin, got {n_9m_notna}"
)
print(f"PASSED 7a: all {len(full_term_9m)} full-term mothers have non-NaN hemoglobin at 9m")

# CHECK 7b: all hemoglobin values are non-negative
assert (full_term_9m[COLUMNS.HEMOGLOBIN_EXPOSURE] >= 0).all(), "Found negative hemoglobin values at 9m"
print("PASSED 7b: all 9-month hemoglobin values are >= 0")

# CHECK 7c: directional - hemorrhage cases have lower mean 9m hemoglobin
no_hemorrhage = full_term_9m[
    (full_term_9m[COLUMNS.POSTPARTUM_HEMORRHAGE] == False)
    & (full_term_9m[COLUMNS.ANTEPARTUM_HEMORRHAGE] == False)
]
any_hemorrhage = full_term_9m[
    (full_term_9m[COLUMNS.POSTPARTUM_HEMORRHAGE] == True)
    | (full_term_9m[COLUMNS.ANTEPARTUM_HEMORRHAGE] == True)
]

mean_no_hem = no_hemorrhage[COLUMNS.HEMOGLOBIN_EXPOSURE].mean()
mean_any_hem = any_hemorrhage[COLUMNS.HEMOGLOBIN_EXPOSURE].mean() if len(any_hemorrhage) else np.nan

if len(any_hemorrhage) > 0:
    assert mean_any_hem < mean_no_hem, (
        f"Expected hemorrhage cases to have lower 9m hemoglobin: "
        f"hemorrhage mean={mean_any_hem:.1f}, no hemorrhage mean={mean_no_hem:.1f}"
    )
    print(f"PASSED 7c: hemorrhage cases have lower 9m hemoglobin ({mean_any_hem:.1f}) vs no hemorrhage ({mean_no_hem:.1f})")
else:
    print("SKIPPED 7c: no hemorrhage cases in this run")

# CHECK 7d: 9m hemoglobin is distinct from pre-9m (0-6w shifted) hemoglobin
# (pipeline modifier replaces with non-pregnant distribution at 9m event)
exact_match_rate = (
    (full_term_9m[COLUMNS.HEMOGLOBIN_EXPOSURE] - pre_9m_hgb.loc[full_term_9m.index]).abs() < 1e-6
).mean()
assert exact_match_rate < 0.05, (
    f"9m hemoglobin matches pre-9m hemoglobin for {exact_match_rate:.1%} of simulants - "
    "expected a new draw from the non-pregnant distribution"
)
print(f"PASSED 7d: 9m hemoglobin differs from 6w hemoglobin ({exact_match_rate:.1%} exact matches)")

# Summary table
rows = []
for label, subset in [('no_hemorrhage', no_hemorrhage), ('pph_only', full_term_9m[
    (full_term_9m[COLUMNS.POSTPARTUM_HEMORRHAGE] == True) & (full_term_9m[COLUMNS.ANTEPARTUM_HEMORRHAGE] == False)
]), ('aph_only', full_term_9m[
    (full_term_9m[COLUMNS.POSTPARTUM_HEMORRHAGE] == False) & (full_term_9m[COLUMNS.ANTEPARTUM_HEMORRHAGE] == True)
]), ('both', full_term_9m[
    (full_term_9m[COLUMNS.POSTPARTUM_HEMORRHAGE] == True) & (full_term_9m[COLUMNS.ANTEPARTUM_HEMORRHAGE] == True)
])]:
    if len(subset) == 0:
        continue
    rows.append({
        'group': label,
        'n': len(subset),
        'mean_6w_hgb': pre_9m_hgb.loc[subset.index].mean(),
        'mean_9m_hgb': subset[COLUMNS.HEMOGLOBIN_EXPOSURE].mean(),
    })

summary_9m = pd.DataFrame(rows)
summary_9m